In [3]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
df_raw = spark.read.csv("Files/Swiggy_Food_Delivery_Dataset.csv", header=True,inferSchema=True)

df_raw.show()

StatementMeta(, eea59b91-db70-4241-a7d3-5e002627667e, 5, Finished, Available, Finished, False)

+--------+-------------------+-------------------+----------------+------------+------------+-------+---------------+----------+-----------+----------+------------+-----------+
|order_id|         order_time|      delivery_time|      restaurant|     cuisine|total_amount| rating|       location|      city|ordered_qty|book_table|online_order|distance_km|
+--------+-------------------+-------------------+----------------+------------+------------+-------+---------------+----------+-----------+----------+------------+-----------+
|ORD_0001|2023-11-19 09:03:00|2023-11-19 09:24:00|            Toit|    Desserts|       58.22|    NEW|Electronic City| bengaluru|          2|        No|         Yes|       7.83|
|ORD_0002|2023-10-25 03:07:00|               NULL|   Meghana Foods|     Mexican|       63.31|  4.0/5|    Marthahalli| bengaluru|          6|       Yes|        NULL|       NULL|
|ORD_0003|2023-10-28 16:48:00|               NULL|        Nandhini|     Mughlai|       70.13|  4.0/5|   Malleshwara

**STEP 2: Data Cleaning (Silver Layer - PySpark)**

In [5]:
from pyspark.sql.functions import col, trim,lower
df = df_raw\
.withColumn("location",trim(lower(col("location")))) \
.withColumn("city",trim(lower(col("city"))))

StatementMeta(, eea59b91-db70-4241-a7d3-5e002627667e, 7, Finished, Available, Finished, False)

In [6]:
df.select("city").distinct().show(truncate=False)

StatementMeta(, eea59b91-db70-4241-a7d3-5e002627667e, 8, Finished, Available, Finished, False)

+---------+
|city     |
+---------+
|bnegaluru|
|bengaluru|
+---------+



In [7]:
df = df.replace(
    ["bnegaluru", "bengaluru"],
    ["bengaluru", "bengaluru"], 
    subset=["city"]
)

StatementMeta(, eea59b91-db70-4241-a7d3-5e002627667e, 9, Finished, Available, Finished, False)

In [8]:
df.select("city").distinct().show(truncate=False)

StatementMeta(, eea59b91-db70-4241-a7d3-5e002627667e, 10, Finished, Available, Finished, False)

+---------+
|city     |
+---------+
|bengaluru|
+---------+



In [13]:
from pyspark.sql.functions import regexp_extract
df = df.withColumn(
    "rating_clean",
    regexp_extract(col("rating"), r'(\d+\.\d+)',1).cast("float")
)

StatementMeta(, eea59b91-db70-4241-a7d3-5e002627667e, 15, Finished, Available, Finished, False)

In [14]:
df=df.fillna({"rating_clean" : 0.0})

StatementMeta(, eea59b91-db70-4241-a7d3-5e002627667e, 16, Finished, Available, Finished, False)

In [16]:
df.select("rating_clean").show()

StatementMeta(, eea59b91-db70-4241-a7d3-5e002627667e, 18, Finished, Available, Finished, False)

+------------+
|rating_clean|
+------------+
|         0.0|
|         4.0|
|         4.0|
|         4.0|
|         0.0|
|         0.0|
|         3.0|
|         3.0|
|         3.1|
|         3.0|
|         3.7|
|         0.0|
|         0.0|
|         2.2|
|         0.0|
|         0.0|
|         2.3|
|         0.0|
|         0.0|
|         4.0|
+------------+
only showing top 20 rows



In [35]:
df.filter(col("rating_clean").isNull()).show()

StatementMeta(, eea59b91-db70-4241-a7d3-5e002627667e, 37, Finished, Available, Finished, False)

+--------+----------+-------------+----------+-------+------------+------+--------+----+-----------+----------+------------+-----------+------------+
|order_id|order_time|delivery_time|restaurant|cuisine|total_amount|rating|location|city|ordered_qty|book_table|online_order|distance_km|rating_clean|
+--------+----------+-------------+----------+-------+------------+------+--------+----+-----------+----------+------------+-----------+------------+
+--------+----------+-------------+----------+-------+------------+------+--------+----+-----------+----------+------------+-----------+------------+



In [36]:
df.filter(col("rating_clean") == "NEW").show()

StatementMeta(, eea59b91-db70-4241-a7d3-5e002627667e, 38, Finished, Available, Finished, False)

+--------+----------+-------------+----------+-------+------------+------+--------+----+-----------+----------+------------+-----------+------------+
|order_id|order_time|delivery_time|restaurant|cuisine|total_amount|rating|location|city|ordered_qty|book_table|online_order|distance_km|rating_clean|
+--------+----------+-------------+----------+-------+------------+------+--------+----+-----------+----------+------------+-----------+------------+
+--------+----------+-------------+----------+-------+------------+------+--------+----+-----------+----------+------------+-----------+------------+



In [18]:
from pyspark.sql.functions import to_timestamp

df = df.withColumn("order_time",to_timestamp("order_time"))
df = df.withColumn("delivery_time",to_timestamp("delivery_time"))

StatementMeta(, eea59b91-db70-4241-a7d3-5e002627667e, 20, Finished, Available, Finished, False)

In [20]:
df.select("delivery_time").show()

StatementMeta(, eea59b91-db70-4241-a7d3-5e002627667e, 22, Finished, Available, Finished, False)

+-------------------+
|      delivery_time|
+-------------------+
|2023-11-19 09:24:00|
|               NULL|
|               NULL|
|               NULL|
|2023-11-20 11:37:00|
|2023-12-09 09:59:00|
|               NULL|
|               NULL|
|2023-12-21 06:59:00|
|               NULL|
|               NULL|
|               NULL|
|               NULL|
|               NULL|
|               NULL|
|               NULL|
|               NULL|
|               NULL|
|2023-11-11 05:46:00|
|               NULL|
+-------------------+
only showing top 20 rows



In [24]:
df.filter(col("delivery_time").isNull()) \
  .select("online_order", "delivery_time") \
  .show(truncate=False)

StatementMeta(, eea59b91-db70-4241-a7d3-5e002627667e, 26, Finished, Available, Finished, False)

+------------+-------------+
|online_order|delivery_time|
+------------+-------------+
|NULL        |NULL         |
|NULL        |NULL         |
|No          |NULL         |
|NULL        |NULL         |
|NULL        |NULL         |
|NULL        |NULL         |
|No          |NULL         |
|NULL        |NULL         |
|NULL        |NULL         |
|NULL        |NULL         |
|No          |NULL         |
|NULL        |NULL         |
|NULL        |NULL         |
|NULL        |NULL         |
|No          |NULL         |
|NULL        |NULL         |
|NULL        |NULL         |
|NULL        |NULL         |
|No          |NULL         |
|No          |NULL         |
+------------+-------------+
only showing top 20 rows



In [25]:
df = df.fillna({
    "online_order" : "No"
})

StatementMeta(, eea59b91-db70-4241-a7d3-5e002627667e, 27, Finished, Available, Finished, False)

In [27]:
df.filter(col("delivery_time").isNull()) \
  .select("online_order", "delivery_time","distance_km") \
  .show(truncate=False)

StatementMeta(, eea59b91-db70-4241-a7d3-5e002627667e, 29, Finished, Available, Finished, False)

+------------+-------------+-----------+
|online_order|delivery_time|distance_km|
+------------+-------------+-----------+
|No          |NULL         |NULL       |
|No          |NULL         |NULL       |
|No          |NULL         |NULL       |
|No          |NULL         |NULL       |
|No          |NULL         |NULL       |
|No          |NULL         |NULL       |
|No          |NULL         |NULL       |
|No          |NULL         |NULL       |
|No          |NULL         |NULL       |
|No          |NULL         |NULL       |
|No          |NULL         |NULL       |
|No          |NULL         |NULL       |
|No          |NULL         |NULL       |
|No          |NULL         |NULL       |
|No          |NULL         |NULL       |
|No          |NULL         |NULL       |
|No          |NULL         |NULL       |
|No          |NULL         |NULL       |
|No          |NULL         |NULL       |
|No          |NULL         |NULL       |
+------------+-------------+-----------+
only showing top

In [28]:
from pyspark.sql.functions import when

df = df.withColumn(
    "delivery_time",
    when(col("online_order")== "No", None).otherwise(col("delivery_time"))
    )

df = df.withColumn(
    "distance_km",
    when(col("online_order") == "No", None).otherwise(col("distance_km"))
)    

StatementMeta(, eea59b91-db70-4241-a7d3-5e002627667e, 30, Finished, Available, Finished, False)

In [30]:
df.select("online_order","delivery_time","distance_km").show()

StatementMeta(, eea59b91-db70-4241-a7d3-5e002627667e, 32, Finished, Available, Finished, False)

+------------+-------------------+-----------+
|online_order|      delivery_time|distance_km|
+------------+-------------------+-----------+
|         Yes|2023-11-19 09:24:00|       7.83|
|          No|               NULL|       NULL|
|          No|               NULL|       NULL|
|          No|               NULL|       NULL|
|         Yes|2023-11-20 11:37:00|      12.34|
|         Yes|2023-12-09 09:59:00|       3.37|
|          No|               NULL|       NULL|
|          No|               NULL|       NULL|
|         Yes|2023-12-21 06:59:00|       9.02|
|          No|               NULL|       NULL|
|          No|               NULL|       NULL|
|          No|               NULL|       NULL|
|          No|               NULL|       NULL|
|          No|               NULL|       NULL|
|          No|               NULL|       NULL|
|          No|               NULL|       NULL|
|          No|               NULL|       NULL|
|          No|               NULL|       NULL|
|         Yes

In [32]:
df = df.fillna({
    "book_table": "No"
})

StatementMeta(, eea59b91-db70-4241-a7d3-5e002627667e, 34, Finished, Available, Finished, False)

In [34]:
df.groupBy("book_table").count().show()

StatementMeta(, eea59b91-db70-4241-a7d3-5e002627667e, 36, Finished, Available, Finished, False)

+------------+-----+
|online_order|count|
+------------+-----+
|          No| 3033|
|         Yes|  967|
+------------+-----+



**STEP 3: Write Clean Data as Delta Table (Lakehouse)**

In [37]:
df.write.mode("overwrite").format("delta").saveAsTable("silver_orders")

StatementMeta(, eea59b91-db70-4241-a7d3-5e002627667e, 39, Finished, Available, Finished, False)

In [39]:
spark.sql("SELECT * from silver_orders LIMIT 10").show()

StatementMeta(, eea59b91-db70-4241-a7d3-5e002627667e, 41, Finished, Available, Finished, False)

+--------+-------------------+-------------------+----------------+------------+------------+-------+---------------+---------+-----------+----------+------------+-----------+------------+
|order_id|         order_time|      delivery_time|      restaurant|     cuisine|total_amount| rating|       location|     city|ordered_qty|book_table|online_order|distance_km|rating_clean|
+--------+-------------------+-------------------+----------------+------------+------------+-------+---------------+---------+-----------+----------+------------+-----------+------------+
|ORD_0001|2023-11-19 09:03:00|2023-11-19 09:24:00|            Toit|    Desserts|       58.22|    NEW|electronic city|bengaluru|          2|        No|         Yes|       7.83|         0.0|
|ORD_0002|2023-10-25 03:07:00|               NULL|   Meghana Foods|     Mexican|       63.31|  4.0/5|    marthahalli|bengaluru|          6|       Yes|          No|       NULL|         4.0|
|ORD_0003|2023-10-28 16:48:00|               NULL|     